# Does LightGBM's cross-conformal really cost more wall-clock than TabPFN's?

This settles **P5**, the one pre-registered prediction in
[tabpfn-conformal](https://github.com/ilyas-elm/tabpfn-conformal) that the project
still reports as unsettled.

**Why it was unsettled.** The main experiments measured TabPFN through the Prior
Labs API, on their GPUs, against LightGBM on a laptop CPU. That is not a race:
most of the TabPFN number was network round-trip. Every wall-clock row in E4
carries `wallclock_comparable: false` for exactly that reason, and the repository
says so rather than quoting the figure.

**What this does.** Both models, one machine, one accelerator, using TabPFN's
downloadable weights so nothing crosses the network while the clock runs. The E4
protocol is unchanged: matched label budgets, the same pool and evaluation
construction, 2 budgets x 3 seeds x 2 families x 2 strategies.

**What it cannot do.** It measures *local* TabPFN, not the managed API, so these
timings do not reproduce the API numbers and are not meant to. The gradient-fit
count is the hardware-independent comparison and does not move either way: 0 for
TabPFN, 1 for LightGBM split, 6 for LightGBM cross at K=5.

## Before running

Right-hand sidebar:

- **Session options, Accelerator**: GPU T4 x2. On CPU this settles nothing, and
  the analysis at the bottom refuses to draw a conclusion.
- **Session options, Internet**: on. It is off by default, and the install and
  the clone both need it.
- **Add-ons, Secrets**: your TabPFN API key, attached to this notebook. Local
  weights need a one-time licence acceptance at <https://ux.priorlabs.ai>
  (Licenses tab), after which the key authorises the download. It authorises
  only the download; inference is local, which is the whole point.
- **+ Add Input**: `Bank Account Fraud Dataset NeurIPS 2022` (Jesus et al.,
  NeurIPS 2022).

Both the GPU and the internet toggle are gated on Kaggle phone verification.

In [ ]:
import subprocess, sys

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("gpu:   ", gpu or "NONE VISIBLE, set the accelerator")
print("python:", sys.version.split()[0])

In [ ]:
import os, pathlib
from kaggle_secrets import UserSecretsClient

# The label is whatever you called the secret; try the likely ones and say
# which matched, so a wrong label costs a second rather than a session.
for label in ("tabpfn v3.5", "TABPFN_TOKEN", "tabpfn_token", "conform"):
    try:
        os.environ["TABPFN_TOKEN"] = UserSecretsClient().get_secret(label)
        print("secret:", label)
        break
    except Exception:
        continue
else:
    raise SystemExit("No TabPFN secret found. Add-ons -> Secrets, and attach "
                     "it to this notebook.")

# Kaggle names the input folder after the dataset slug, and some datasets nest
# a level down, so find the file rather than assume the path.
base = next(pathlib.Path("/kaggle/input").rglob("Base.csv"), None)
if base is None:
    attached = [p.name for p in pathlib.Path("/kaggle/input").glob("*")]
    raise SystemExit(f"Base.csv not found. Inputs attached: {attached or 'none'}. "
                     "Sidebar -> + Add Input.")
DATA = str(base.parent)
print("data:  ", DATA)

## The code being measured

Cloned from the repository rather than pasted here, so the notebook cannot drift
from the library it is supposed to be measuring. `wallclock.py` is 200 lines and
readable in the repo.

In [ ]:
%cd /kaggle/working
!pip install -q tabpfn lightgbm
!rm -rf /kaggle/working/tabpfn-conformal
!git clone -q https://github.com/ilyas-elm/tabpfn-conformal.git
%cd /kaggle/working/tabpfn-conformal
!pip install -q -e .

## The measurement

Both families are warmed up before anything is timed. TabPFN does not load its
weights until `fit`, and on a fresh machine that first call also *downloads*
them, 876 MB here. Timing it would have charged TabPFN a large network cost in
the one experiment whose entire purpose is to have no network in it.

The JSON is written after every configuration, so a session that dies part-way
still leaves usable rows. Roughly half an hour on a T4.

In [ ]:
!python experiments/kaggle/wallclock.py --data "$DATA"

## The verdict

The seeds are shared between the two families, so the comparison is paired
rather than a difference of means, and it reports the standard error and the
number of seeds behind it.

In [ ]:
!python experiments/analyze_kaggle.py

## Reading it

Whatever the timings say, the claim the repository leads with is the
hardware-independent one: **0 gradient-trained fits against LightGBM's 6** at
K=5. That is why K-fold cross-conformal is affordable on TabPFN at all, and no
choice of machine changes it.

The wall-clock figure is the secondary question, and P5 predicted it would
favour TabPFN. The API-based measurement suggested the opposite but could not be
trusted, because of the confound above. This notebook is what replaces it.

Full method, benchmarks and the falsification table:
<https://github.com/ilyas-elm/tabpfn-conformal>